# TP 5 — LSTM Forecasting de rendement agricole avec **Keras/TensorFlow**

*Notebook miroir de TP 6 (PyTorch) — même cas d'usage, même dataset, API différente.*

## Introduction

Les séries temporelles agricoles (rendement par année, état, saison) présentent des tendances
et saisonnalités que les modèles classiques capturent mal. Le **Long Short-Term Memory (LSTM)**
est une architecture de réseau de neurones récurrent conçue pour mémoriser des dépendances
à long terme — ce qui en fait un choix naturel pour le **forecasting de production agricole**.

**Objectifs pédagogiques :**
- Préparer des séquences temporelles (sliding window)
- Construire et entraîner un LSTM avec Keras (`Sequential` API)
- Utiliser `EarlyStopping` et `ModelCheckpoint`
- Visualiser les prédictions vs valeurs réelles

**Point de comparaison Keras vs PyTorch (cf. TP 6) :**
En Keras, la boucle d'entraînement est encapsulée dans `model.fit()` avec des callbacks.
En PyTorch (TP 6), elle est entièrement manuelle : `forward -> loss -> backward -> optimizer.step()`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

import warnings
warnings.filterwarnings('ignore')

print('TensorFlow version:', tf.__version__)

## 1. Chargement et préparation des données

In [ ]:
# Chargement du dataset principal
df = pd.read_excel('Data/crop_csv_file.xlsx')
print('Shape:', df.shape)
print('Colonnes:', list(df.columns))
df.head(3)

## 2. Construction des séries temporelles

On agrège la **production totale par année** pour obtenir une série temporelle
continue. La variable cible est `Production` (en tonnes), normalisée avec MinMaxScaler.

In [ ]:
# Agrégation : production totale par année
df_agg = df.groupby('Crop_Year')['Production'].sum().reset_index()
df_agg.columns = ['Year', 'Production']
df_agg = df_agg.sort_values('Year').reset_index(drop=True)
print(df_agg)

# Visualisation de la série temporelle
plt.figure(figsize=(12, 4))
plt.plot(df_agg['Year'], df_agg['Production'], marker='o', linewidth=2, color='#2ecc71')
plt.title('Production agricole totale par année (1997-2014)', fontsize=14)
plt.xlabel('Année'); plt.ylabel('Production (tonnes)')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 3. Sliding Window — création des séquences

Pour alimenter le LSTM, on transforme la série 1D en séquences entrée/sortie :
- `window_size = 3` : on utilise 3 années consécutives pour prédire la 4e
- Normalisation MinMaxScaler sur [0, 1]
- Reshape en `(samples, timesteps, features)` pour le LSTM

In [ ]:
def create_sequences(series, window_size):
    """Cree des paires (X, y) pour apprentissage supervise sur serie temporelle."""
    X, y = [], []
    for i in range(len(series) - window_size):
        X.append(series[i:i + window_size])
        y.append(series[i + window_size])
    return np.array(X), np.array(y)

# Normalisation
scaler = MinMaxScaler()
production_scaled = scaler.fit_transform(df_agg[['Production']]).flatten()

# Creation des sequences
WINDOW_SIZE = 3
X, y = create_sequences(production_scaled, WINDOW_SIZE)

# Reshape pour LSTM : (samples, timesteps, features)
X = X.reshape((X.shape[0], X.shape[1], 1))
print('X shape:', X.shape, ' | y shape:', y.shape)

# Split train/test (80/20 chronologique)
split = int(len(X) * 0.8)
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]
print('Train:', X_train.shape, ' | Test:', X_test.shape)

## 4. Architecture LSTM (Keras)

Architecture : **LSTM(64) -> Dropout -> LSTM(32) -> Dense(1)**

```
Input (window_size=3, features=1)
  |
  v
LSTM(64, return_sequences=True)  <- memorise la sequence complete
  |
Dropout(0.2)
  |
LSTM(32)                         <- condense en vecteur d'etat
  |
Dense(1)                         <- prediction scalaire (production normalisee)
```

**API Keras** : `model.compile()` + `model.fit()` avec callbacks.
Contraste avec la boucle manuelle PyTorch dans TP 6.

In [ ]:
# Definition du modele LSTM avec Keras Sequential
model = Sequential([
    LSTM(64, return_sequences=True, input_shape=(WINDOW_SIZE, 1)),
    Dropout(0.2),
    LSTM(32, return_sequences=False),
    Dropout(0.2),
    Dense(1)
], name='LSTM_Crop_Forecasting_Keras')

model.compile(optimizer='adam', loss='mse', metrics=['mae'])
model.summary()

## 5. Entrainement avec callbacks Keras

- `EarlyStopping` : stoppe si la val_loss ne s'ameliore plus pendant 20 epochs
- `ModelCheckpoint` : sauvegarde le meilleur modele automatiquement

> **Difference Keras vs PyTorch** : En Keras, ces mecanismes sont des objets
> `Callback` passes a `fit()`. En PyTorch (TP 6), ils sont implementes
> manuellement dans la boucle d'entrainement (condition `if val_loss < best`).

In [ ]:
# Callbacks
early_stop = EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True)
checkpoint = ModelCheckpoint('best_lstm_keras.h5', monitor='val_loss',
                             save_best_only=True, verbose=0)

# Entrainement — la boucle est geree par Keras
history = model.fit(
    X_train, y_train,
    epochs=200,
    batch_size=4,
    validation_split=0.2,
    callbacks=[early_stop, checkpoint],
    verbose=1
)

In [ ]:
# Courbes de loss
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(history.history['loss'], label='Train Loss', color='#3498db')
axes[0].plot(history.history['val_loss'], label='Val Loss', color='#e74c3c')
axes[0].set_title('MSE Loss — LSTM Keras')
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(history.history['mae'], label='Train MAE', color='#2ecc71')
axes[1].plot(history.history['val_mae'], label='Val MAE', color='#e67e22')
axes[1].set_title('MAE — LSTM Keras')
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout(); plt.show()
print('Epochs effectuees :', len(history.history['loss']))

## 6. Evaluation et visualisation des predictions

In [ ]:
# Predictions sur le jeu de test
y_pred_scaled = model.predict(X_test).flatten()

# Denormalisation
y_pred = scaler.inverse_transform(y_pred_scaled.reshape(-1, 1)).flatten()
y_true = scaler.inverse_transform(y_test.reshape(-1, 1)).flatten()

# Metriques
mae = mean_absolute_error(y_true, y_pred)
rmse = mean_squared_error(y_true, y_pred, squared=False)
print('MAE  : {:,.0f} tonnes'.format(mae))
print('RMSE : {:,.0f} tonnes'.format(rmse))

# Visualisation predictions vs realite
test_years = df_agg['Year'].values[WINDOW_SIZE + split:]
train_years = df_agg['Year'].values[WINDOW_SIZE:WINDOW_SIZE + split]
y_train_orig = scaler.inverse_transform(y_train.reshape(-1, 1)).flatten()

plt.figure(figsize=(12, 5))
plt.plot(train_years, y_train_orig, label='Train', color='#95a5a6', linewidth=1.5)
plt.plot(test_years, y_true, label='Realite (test)', color='#2ecc71', linewidth=2, marker='o')
plt.plot(test_years, y_pred, label='Prediction LSTM Keras',
         color='#e74c3c', linewidth=2, marker='s', linestyle='--')
plt.title('Forecasting de production agricole — LSTM Keras', fontsize=14)
plt.xlabel('Annee'); plt.ylabel('Production (tonnes)')
plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 7. Resume comparatif Keras vs PyTorch

| Aspect | Keras (ce notebook) | PyTorch (TP 6) |
|---|---|---|
| **Definition LSTM** | `LSTM(64, return_sequences=True)` | `nn.LSTM(1, 64, num_layers=2, batch_first=True)` |
| **Etat cache** | Gere automatiquement | Retourne explicitement `(output, (h_n, c_n))` |
| **Entrainement** | `model.fit(X, y, callbacks=[...])` | Boucle `for epoch in range(...)` manuelle |
| **Early stopping** | `EarlyStopping` callback integre | Condition `if val_loss < best_loss: ...` a coder |
| **Sauvegarde** | `ModelCheckpoint` callback | `torch.save(model.state_dict(), path)` |
| **Flexibilite** | API haut niveau, prototypage rapide | Controle total sur le gradient et le forward pass |